In [6]:
!pip show chromadb rank_bm25 ragas groq | grep -E "Name|Version"

In [7]:
!pip install -q chromadb rank_bm25 "ragas==0.4.3" "langchain-community==0.3.24" "langchain-core==0.3.59" "langchain-groq==0.3.2" "langchain-openai==0.3.16"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
litellm 1.82.4 requires openai>=2.8.0, but you have openai 1.109.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0

In [8]:
import os
import shutil

src = '/kaggle/input/datasets/harshamin07/researchlens-dataset'
dst = '/kaggle/working/'

os.makedirs(dst, exist_ok=True)

for fname in ['bm25_index.pkl', 'bm25_metadata.pkl', 'fixed_chunks.pkl',
              'semantic_chunks.pkl', 'parsed_papers.json',
              'papers_metadata.json', 'qasper_pairs.json']:
    src_path = os.path.join(src, fname)
    if os.path.exists(src_path):
        shutil.copy(src_path, os.path.join(dst, fname))
        print(f"  ✓ {fname}")

chroma_src = os.path.join(src, 'indexes', 'chroma_db')
chroma_dst = os.path.join(dst, 'chroma_db')
if os.path.exists(chroma_src) and not os.path.exists(chroma_dst):
    shutil.copytree(chroma_src, chroma_dst)
    print(f"  ✓ chroma_db/")
elif os.path.exists(os.path.join(src, 'chroma_db')) and not os.path.exists(chroma_dst):
    shutil.copytree(os.path.join(src, 'chroma_db'), chroma_dst)
    print(f"  ✓ chroma_db/ (root)")

contradiction_src = '/kaggle/input/datasets/harshamin07/researchlens-contradiction/contradiction_results.json'
if os.path.exists(contradiction_src):
    shutil.copy(contradiction_src, os.path.join(dst, 'contradiction_results.json'))
    print(f"  ✓ contradiction_results.json")

print("\nDone loading from dataset")

  ✓ bm25_index.pkl
  ✓ bm25_metadata.pkl
  ✓ fixed_chunks.pkl
  ✓ semantic_chunks.pkl
  ✓ parsed_papers.json
  ✓ papers_metadata.json
  ✓ qasper_pairs.json
  ✓ contradiction_results.json

Done loading from dataset


In [9]:
import warnings
warnings.filterwarnings('ignore')
import pickle
import json
import time
import numpy as np
import pandas as pd
from abc import ABC, abstractmethod
from typing import List, Dict, Tuple
import torch
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer, AutoModelForCausalLM
from kaggle_secrets import UserSecretsClient
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference
)

In [10]:
with open('/kaggle/working/bm25_index.pkl', 'rb') as f:
    bm25_index = pickle.load(f)
with open('/kaggle/working/bm25_metadata.pkl', 'rb') as f:
    bm25_metadata = pickle.load(f)

chroma_client = chromadb.PersistentClient(path='/kaggle/working/chroma_db')
fixed_collection = chroma_client.get_collection("fixed_chunks")
semantic_collection = chroma_client.get_collection("semantic_chunks")

embed_model = SentenceTransformer('all-MiniLM-L6-v2')
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print(f"BM25: {len(bm25_metadata)} docs")
print(f"ChromaDB fixed: {fixed_collection.count()}")
print(f"ChromaDB semantic: {semantic_collection.count()}")
print(f"Models loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

BM25: 3205 docs
ChromaDB fixed: 3205
ChromaDB semantic: 3427
Models loaded


In [11]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer.pad_token = tokenizer.eos_token
print(f"Mistral loaded on: {next(llm.parameters()).device}")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Mistral loaded on: cuda:0


In [12]:
secrets = UserSecretsClient()
groq_api_key = secrets.get_secret("GROQ_API_KEY")

groq_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=groq_api_key,
    temperature=0
)

evaluator_llm = LangchainLLMWrapper(groq_llm)
print("Groq evaluator LLM ready")

Groq evaluator LLM ready


In [13]:
class BaseRetriever(ABC):
    @abstractmethod
    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        pass

    @property
    @abstractmethod
    def strategy_name(self) -> str:
        pass

    def _make_result(self, text, score, paper_id, title, chunk_id):
        return {'text': text, 'score': float(score), 'paper_id': paper_id,
                'title': title, 'chunk_id': chunk_id}

class NaiveRetriever(BaseRetriever):
    @property
    def strategy_name(self): return "naive_rag"
    def retrieve(self, query, top_k=5):
        qe = embed_model.encode(query).tolist()
        r = fixed_collection.query(query_embeddings=[qe], n_results=top_k,
                                   include=['documents', 'distances', 'metadatas'])
        return [self._make_result(d, 1-s, m['paper_id'], m['title'], m.get('chunk_id',''))
                for d, s, m in zip(r['documents'][0], r['distances'][0], r['metadatas'][0])]

class SemanticRetriever(BaseRetriever):
    @property
    def strategy_name(self): return "semantic_rag"
    def retrieve(self, query, top_k=5):
        qe = embed_model.encode(query).tolist()
        r = semantic_collection.query(query_embeddings=[qe], n_results=top_k,
                                      include=['documents', 'distances', 'metadatas'])
        return [self._make_result(d, 1-s, m['paper_id'], m['title'], m.get('chunk_id',''))
                for d, s, m in zip(r['documents'][0], r['distances'][0], r['metadatas'][0])]

class HybridRetriever(BaseRetriever):
    @property
    def strategy_name(self): return "hybrid_rag"
    def retrieve(self, query, top_k=5):
        fetch_k = top_k * 4
        qe = embed_model.encode(query).tolist()
        dr = fixed_collection.query(query_embeddings=[qe], n_results=fetch_k,
                                    include=['documents', 'distances', 'metadatas'])
        dense_hits = {}
        for rank, (d, s, m) in enumerate(zip(dr['documents'][0], dr['distances'][0], dr['metadatas'][0])):
            cid = m.get('chunk_id', f"d_{rank}")
            dense_hits[cid] = {'rank': rank, 'text': d, 'score': 1-s,
                               'paper_id': m['paper_id'], 'title': m['title']}
        bm25_scores = bm25_index.get_scores(query.lower().split())
        top_idx = np.argsort(bm25_scores)[::-1][:fetch_k]
        bm25_hits = {}
        for rank, idx in enumerate(top_idx):
            cid = bm25_metadata[idx]['chunk_id']
            bm25_hits[cid] = {'rank': rank, 'text': bm25_metadata[idx]['text'],
                              'score': bm25_scores[idx], 'paper_id': bm25_metadata[idx]['paper_id'],
                              'title': bm25_metadata[idx]['title']}
        k = 60
        all_ids = set(dense_hits) | set(bm25_hits)
        rrf = {cid: (1/(dense_hits[cid]['rank']+k) if cid in dense_hits else 0) +
                    (1/(bm25_hits[cid]['rank']+k) if cid in bm25_hits else 0)
               for cid in all_ids}
        top_ids = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [self._make_result(
            (dense_hits.get(cid) or bm25_hits[cid])['text'], rrf[cid],
            (dense_hits.get(cid) or bm25_hits[cid])['paper_id'],
            (dense_hits.get(cid) or bm25_hits[cid])['title'], cid)
            for cid in top_ids]

class HybridRerankerRetriever(BaseRetriever):
    @property
    def strategy_name(self): return "hybrid_rerank"
    def retrieve(self, query, top_k=5):
        candidates = HybridRetriever().retrieve(query, top_k=20)
        scores = reranker.predict([[query, c['text']] for c in candidates])
        for c, s in zip(candidates, scores):
            c['score'] = float(s)
        return sorted(candidates, key=lambda x: x['score'], reverse=True)[:top_k]

QUERY_SIGNALS = {
    'consensus': ['do papers agree', 'do researchers agree', 'is there consensus',
                  'controversy', 'debate', 'conflicting', 'disagree',
                  'contradiction', 'contradictory', 'dispute'],
    'comparative': ['compare', 'difference between', 'differ', 'versus', ' vs ',
                    'better than', 'worse than', 'advantage', 'disadvantage',
                    'trade-off', 'tradeoff', 'which is', 'contrast'],
    'procedural': ['how to', 'how do i', 'steps to', 'implement', 'build',
                   'create', 'set up', 'train', 'fine-tune', 'walkthrough'],
    'factual': ['what is', 'what are', 'define', 'definition', 'explain',
                'describe', 'who is', 'when was', 'where is', 'which paper']
}

QUERY_TO_STRATEGY = {
    'factual': 'naive_rag',
    'comparative': 'hybrid_rerank',
    'consensus': 'hybrid_rag',
    'procedural': 'semantic_rag'
}

STRATEGY_TO_RETRIEVER = {
    'naive_rag': NaiveRetriever(),
    'semantic_rag': SemanticRetriever(),
    'hybrid_rag': HybridRetriever(),
    'hybrid_rerank': HybridRerankerRetriever()
}

def keyword_classify(query):
    q = query.lower()
    for qtype, signals in QUERY_SIGNALS.items():
        for signal in signals:
            if signal in q:
                return qtype, 0.95
    return 'factual', 0.60

def route_query(query):
    qtype, confidence = keyword_classify(query)
    strategy = QUERY_TO_STRATEGY[qtype]
    return {'query_type': qtype, 'strategy': strategy,
            'confidence': confidence, 'retriever': STRATEGY_TO_RETRIEVER[strategy]}

print("All retriever and router classes defined")

All retriever and router classes defined


In [14]:
def build_prompt(query: str, chunks: List[Dict]) -> str:
    context = "\n\n".join([
        f"[Source: {c['title'][:60]}]\n{c['text']}"
        for c in chunks
    ])
    return f"""[INST] You are a research assistant. Answer the question using ONLY the provided context.
If the context does not contain enough information, say "I don't have enough information."
Be concise. Maximum 3 sentences.

Context:
{context}

Question: {query} [/INST]"""

def generate_answer(query: str, chunks: List[Dict], max_new_tokens: int = 150) -> str:
    prompt = build_prompt(query, chunks)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=3000).to(llm.device)
    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

print("Generation function defined")

Generation function defined


In [15]:
with open('/kaggle/working/qasper_pairs.json', 'r') as f:
    qasper_pairs = json.load(f)

eval_questions = qasper_pairs[:50]

strategies = [
    NaiveRetriever(),
    SemanticRetriever(),
    HybridRetriever(),
    HybridRerankerRetriever()
]

strategy_results = {s.strategy_name: [] for s in strategies}
strategy_latencies = {s.strategy_name: [] for s in strategies}

print(f"Evaluating {len(eval_questions)} questions across {len(strategies)} strategies...")
print("This will take 20-30 minutes — each question runs 4 retrievals + 4 generations\n")

for i, pair in enumerate(eval_questions):
    query = pair['question']
    reference = pair['answer']

    if i % 10 == 0:
        print(f"Progress: {i}/{len(eval_questions)}")

    for strategy in strategies:
        start = time.time()
        chunks = strategy.retrieve(query, top_k=5)
        answer = generate_answer(query, chunks)
        latency = time.time() - start

        strategy_results[strategy.strategy_name].append({
            'user_input': query,
            'response': answer,
            'retrieved_contexts': [c['text'] for c in chunks],
            'reference': reference
        })
        strategy_latencies[strategy.strategy_name].append(latency)

print(f"\nDone. Generated {len(eval_questions) * len(strategies)} answers total.")

Evaluating 50 questions across 4 strategies...
This will take 20-30 minutes — each question runs 4 retrievals + 4 generations

Progress: 0/50
Progress: 10/50
Progress: 20/50
Progress: 30/50
Progress: 40/50

Done. Generated 200 answers total.


In [16]:
import re
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

def compute_faithfulness(answer: str, chunks: List[Dict]) -> float:
    if not answer or "don't have enough information" in answer.lower():
        return 0.0
    context_text = ' '.join([c['text'] for c in chunks])
    context_words = set(re.sub(r'[^\w\s]', '', context_text.lower()).split())
    answer_words = set(re.sub(r'[^\w\s]', '', answer.lower()).split())
    stopwords = {'the','a','an','is','are','was','were','in','on','at','to',
                 'for','of','and','or','but','it','this','that','be','have',
                 'has','had','with','from','by','as','we','our','their','its'}
    content_words = answer_words - stopwords
    if not content_words:
        return 0.0
    return len(content_words & context_words) / len(content_words)

def compute_context_precision(query: str, chunks: List[Dict]) -> float:
    query_emb = embed_model.encode(query, convert_to_numpy=True)
    chunk_embs = embed_model.encode([c['text'] for c in chunks], convert_to_numpy=True)
    sims = sklearn_cosine([query_emb], chunk_embs)[0]
    relevant = sum(1 for s in sims if s > 0.4)
    return relevant / len(chunks) if chunks else 0.0

def compute_answer_relevance(query: str, answer: str) -> float:
    if not answer or "don't have enough information" in answer.lower():
        return 0.0
    query_emb = embed_model.encode(query, convert_to_numpy=True)
    answer_emb = embed_model.encode(answer, convert_to_numpy=True)
    return float(sklearn_cosine([query_emb], [answer_emb])[0][0])

ragas_scores = {}

for strategy_name, results in strategy_results.items():
    print(f"Evaluating {strategy_name}...")
    
    faithfulness_scores = []
    precision_scores = []
    relevance_scores = []
    
    for r in results:
        chunks = [{'text': t} for t in r['retrieved_contexts']]
        faithfulness_scores.append(compute_faithfulness(r['response'], chunks))
        precision_scores.append(compute_context_precision(r['user_input'], chunks))
        relevance_scores.append(compute_answer_relevance(r['user_input'], r['response']))
    
    ragas_scores[strategy_name] = {
        'faithfulness': round(np.mean(faithfulness_scores), 3),
        'context_precision': round(np.mean(precision_scores), 3),
        'answer_relevance': round(np.mean(relevance_scores), 3),
        'avg_latency_ms': round(np.mean(strategy_latencies[strategy_name]) * 1000)
    }
    
    print(f"  ✓ faithfulness={ragas_scores[strategy_name]['faithfulness']} | "
          f"precision={ragas_scores[strategy_name]['context_precision']} | "
          f"relevance={ragas_scores[strategy_name]['answer_relevance']} | "
          f"latency={ragas_scores[strategy_name]['avg_latency_ms']}ms")

print("\nDone.")

Evaluating naive_rag...
  ✓ faithfulness=0.666 | precision=0.688 | relevance=0.359 | latency=17022ms
Evaluating semantic_rag...
  ✓ faithfulness=0.647 | precision=0.66 | relevance=0.411 | latency=17863ms
Evaluating hybrid_rag...
  ✓ faithfulness=0.701 | precision=0.364 | relevance=0.341 | latency=17656ms
Evaluating hybrid_rerank...
  ✓ faithfulness=0.727 | precision=0.384 | relevance=0.356 | latency=19418ms

Done.


In [17]:
print("Evaluating adaptive router...")
router_results = []
router_latencies = []
router_strategy_counts = {}

for i, pair in enumerate(eval_questions):
    query = pair['question']
    reference = pair['answer']
    
    routing = route_query(query)
    retriever = routing['retriever']
    
    start = time.time()
    chunks = retriever.retrieve(query, top_k=5)
    answer = generate_answer(query, chunks)
    latency = time.time() - start
    
    router_results.append({
        'user_input': query,
        'response': answer,
        'retrieved_contexts': [c['text'] for c in chunks],
        'reference': reference,
        'strategy_used': routing['strategy']
    })
    router_latencies.append(latency)
    
    s = routing['strategy']
    router_strategy_counts[s] = router_strategy_counts.get(s, 0) + 1
    
    if i % 10 == 0:
        print(f"  Progress: {i}/{len(eval_questions)}")

faithfulness_scores = []
precision_scores = []
relevance_scores = []

for r in router_results:
    chunks = [{'text': t} for t in r['retrieved_contexts']]
    faithfulness_scores.append(compute_faithfulness(r['response'], chunks))
    precision_scores.append(compute_context_precision(r['user_input'], chunks))
    relevance_scores.append(compute_answer_relevance(r['user_input'], r['response']))

ragas_scores['adaptive_router'] = {
    'faithfulness': round(np.mean(faithfulness_scores), 3),
    'context_precision': round(np.mean(precision_scores), 3),
    'answer_relevance': round(np.mean(relevance_scores), 3),
    'avg_latency_ms': round(np.mean(router_latencies) * 1000)
}

print(f"\nAdaptive router:")
print(f"  faithfulness={ragas_scores['adaptive_router']['faithfulness']} | "
      f"precision={ragas_scores['adaptive_router']['context_precision']} | "
      f"relevance={ragas_scores['adaptive_router']['answer_relevance']} | "
      f"latency={ragas_scores['adaptive_router']['avg_latency_ms']}ms")
print(f"  Strategy distribution: {router_strategy_counts}")

Evaluating adaptive router...
  Progress: 0/50
  Progress: 10/50
  Progress: 20/50
  Progress: 30/50
  Progress: 40/50

Adaptive router:
  faithfulness=0.697 | precision=0.652 | relevance=0.356 | latency=17366ms
  Strategy distribution: {'naive_rag': 45, 'hybrid_rerank': 5}


In [18]:
rows = []
display_names = {
    'naive_rag': 'A · Naive RAG',
    'semantic_rag': 'B · Semantic RAG',
    'hybrid_rag': 'C · Hybrid RAG',
    'hybrid_rerank': 'D · Hybrid + Reranker',
    'adaptive_router': 'Adaptive Router'
}

for strategy_name, scores in ragas_scores.items():
    if 'error' in scores:
        continue
    rows.append({
        'Strategy': display_names.get(strategy_name, strategy_name),
        'Faithfulness': scores['faithfulness'],
        'Context Precision': scores['context_precision'],
        'Answer Relevance': scores['answer_relevance'],
        'Avg Latency (ms)': scores['avg_latency_ms']
    })

results_df = pd.DataFrame(rows).set_index('Strategy')

print("\n" + "="*75)
print("RESEARCHLENS — STRATEGY COMPARISON TABLE")
print("="*75)
print(results_df.to_string())
print("="*75)
print(f"\nEvaluated on {len(eval_questions)} QASPER questions")
print(f"Metrics: custom implementation (faithfulness=token overlap, "
      f"precision=embedding similarity, relevance=query-answer similarity)")
print(f"Generator: Mistral-7B-Instruct-v0.2")


RESEARCHLENS — STRATEGY COMPARISON TABLE
                       Faithfulness  Context Precision  Answer Relevance  Avg Latency (ms)
Strategy                                                                                  
A · Naive RAG                 0.666              0.688             0.359             17022
B · Semantic RAG              0.647              0.660             0.411             17863
C · Hybrid RAG                0.701              0.364             0.341             17656
D · Hybrid + Reranker         0.727              0.384             0.356             19418
Adaptive Router               0.697              0.652             0.356             17366

Evaluated on 50 QASPER questions
Metrics: custom implementation (faithfulness=token overlap, precision=embedding similarity, relevance=query-answer similarity)
Generator: Mistral-7B-Instruct-v0.2


In [19]:
print("CUSTOM METRICS SUMMARY")
print("-"*50)
print(f"Routing Accuracy: 100% on 40-query labeled test set (Notebook 04)")

valid_scores = {k: v for k, v in ragas_scores.items() if 'error' not in v and k != 'adaptive_router'}
best_faith_strategy = max(valid_scores, key=lambda x: valid_scores[x]['faithfulness'])
naive_faith = ragas_scores['naive_rag']['faithfulness']
best_faith = ragas_scores[best_faith_strategy]['faithfulness']

if naive_faith > 0:
    improvement = ((best_faith - naive_faith) / naive_faith) * 100
    print(f"Best Faithfulness:    {best_faith} ({display_names[best_faith_strategy]})")
    print(f"Improvement vs Naive: +{improvement:.1f}%")

naive_lat = ragas_scores['naive_rag']['avg_latency_ms']
rerank_lat = ragas_scores['hybrid_rerank']['avg_latency_ms']
overhead = ((rerank_lat - naive_lat) / naive_lat) * 100
print(f"Reranker overhead:    +{overhead:.0f}% latency vs Naive")
print(f"Router distribution:  {router_strategy_counts}")

CUSTOM METRICS SUMMARY
--------------------------------------------------
Routing Accuracy: 100% on 40-query labeled test set (Notebook 04)
Best Faithfulness:    0.727 (D · Hybrid + Reranker)
Improvement vs Naive: +9.2%
Reranker overhead:    +14% latency vs Naive
Router distribution:  {'naive_rag': 45, 'hybrid_rerank': 5}


In [21]:
results_df.to_csv('/kaggle/working/comparison_table.csv')

full_results = {
    'ragas_scores': ragas_scores,
    'strategy_latencies': {k: [round(v*1000) for v in vals]
                           for k, vals in strategy_latencies.items()},
    'router_strategy_distribution': router_strategy_counts,
    'eval_size': len(eval_questions),
    'metrics': 'custom (faithfulness=token overlap, precision=embedding similarity, relevance=query-answer cosine)',
    'generator': 'Mistral-7B-Instruct-v0.2',
    'summary': {
        'best_faithfulness': 0.727,
        'best_faithfulness_strategy': 'hybrid_rerank',
        'improvement_vs_naive': '9.2%',
        'reranker_latency_overhead': '14%',
        'routing_accuracy': '100% on 40-query test set'
    }
}

with open('/kaggle/working/full_eval_results.json', 'w') as f:
    json.dump(full_results, f, indent=2)

print("Saved:")
print("  ✓ comparison_table.csv")
print("  ✓ full_eval_results.json")
print("\n=== Notebook 07 Complete ===")
print("\nFINAL RESULTS:")
print(results_df.to_string())

Saved:
  ✓ comparison_table.csv
  ✓ full_eval_results.json

=== Notebook 07 Complete ===

FINAL RESULTS:
                       Faithfulness  Context Precision  Answer Relevance  Avg Latency (ms)
Strategy                                                                                  
A · Naive RAG                 0.666              0.688             0.359             17022
B · Semantic RAG              0.647              0.660             0.411             17863
C · Hybrid RAG                0.701              0.364             0.341             17656
D · Hybrid + Reranker         0.727              0.384             0.356             19418
Adaptive Router               0.697              0.652             0.356             17366
